# Nier2iOS — NieR:Automata 1.0.3 PT-BR
Execute as células em ordem. Links temporários são solicitados localmente, não armazenados no GitHub. Nenhum workflow é disparado até a última célula.

In [ ]:
# Nier2iOS - run cells in order; no signed URLs are committed to GitHub.
from pathlib import Path
import os, sys, subprocess, getpass, importlib, argparse
from google.colab import auth
auth.authenticate_user()
repo = Path("/content/Nier2iOS-repo")
if not (repo/".git").is_dir():
    subprocess.run(["git","clone","--depth","1","--branch","Nier2iOS","https://github.com/joao9vegeto-stack/KintTany.git",str(repo)],check=True)
else:
    subprocess.run(["git","-C",str(repo),"fetch","--depth","1","origin","Nier2iOS"],check=True)
    subprocess.run(["git","-C",str(repo),"reset","--hard","FETCH_HEAD"],check=True)
subprocess.run([sys.executable,"-m","pip","-q","install","requests","google-api-python-client","google-auth"],check=True)
sys.path.insert(0,str(repo/"nier/colab"))
sys.path.insert(0,str(repo/"nier"))
import run_colab, archive_ingest
importlib.reload(run_colab); importlib.reload(archive_ingest)
api = run_colab.drive_api()
PARENT = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"
args=argparse.Namespace(drive_folder_id=PARENT)


In [ ]:
# Only ask for links that have not already been ingested.
from source_archive import GAME_FOLDER_ID, list_children
present={x["name"].lower() for x in list_children(api,GAME_FOLDER_ID)}
def source_url(label, needed):
    return getpass.getpass(label).strip() if needed else ""
urls=[source_url(f"URL base RAR parte {i}: ", "base.xci" not in present) for i in range(1,4)]
update=source_url("URL update RAR: ","update.nsp" not in present)
dlcs=source_url("URL DLC RAR: ","dlc-inventory.json" not in present)
translation=source_url("URL mod PT-BR ZIP: ","translation.zip" not in present)
archive_ingest.ingest(api,PARENT,urls,update,dlcs,translation)


In [ ]:
# Recover only a verified existing handoff or perform heavy Linux preparation.
handoff=run_colab.recover(args,api)
if not handoff:
    handoff=run_colab.prepare(args,api)


In [ ]:
# Only this cell triggers the Nier2iOS workflow.
os.environ["NIER_GITHUB_TOKEN"]=getpass.getpass("GitHub PAT (KintTany Contents read/write): ").strip()
try:
    run_colab.dispatch(argparse.Namespace(handoff=Path("/content/nier-handoff.json")))
finally:
    os.environ.pop("NIER_GITHUB_TOKEN",None)
